# EEG 3C + 3D — Clean Exact 3B Pipeline

This version starts fresh from `3B_Exact_40_25_Split`. 3C converts every complete 24-row epoch to one sample, fits **separate PCA per subject**, and saves those PCA outputs to Drive. 3D loads the saved PCA outputs and fits separate KNN, Logistic Regression, and SVM models per subject. It reports accuracy, MAE, Spearman, and saves target/prediction for every epoch.


In [ ]:
!pip -q install pandas numpy scipy scikit-learn joblib
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
import importlib
from pathlib import Path

CODE_DIR = '/content/drive/MyDrive/EEG Shared/Code/Code Luke'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import eeg_3c_exact3b_saved_pca
import eeg_3d_models_from_saved_pca
importlib.reload(eeg_3c_exact3b_saved_pca)
importlib.reload(eeg_3d_models_from_saved_pca)

from eeg_3c_exact3b_saved_pca import build_and_save_pca_from_exact3b
from eeg_3d_models_from_saved_pca import run_models_from_saved_pca, combine_all_six_predictions

print('3C + 3D functions loaded.')


3C + 3D functions loaded.


In [ ]:
SPLIT_FOLDER = Path(
    '/content/drive/MyDrive/EEG Shared/Processed_data/'
    'Kosachenko_All_Subjects/3B_Exact_40_25_Split'
)

PCA_OUTPUT_FOLDER = Path(
    '/content/drive/MyDrive/EEG Shared/Processed_data/'
    'Kosachenko_All_Subjects/3C_PCA_Outputs_Clean'
)

RESULTS_FOLDER = Path(
    '/content/drive/MyDrive/EEG Shared/Results/Luke/'
    '3D_Models_From_Clean_PCA'
)

TARGET_COLUMN = 'load_in_sequence'
EPOCH_ROWS = 24
PC_COUNT = 20
CHUNK_ROWS = 50000

# Keep False while developing/tuning. Change to True only when you are ready
# to process the final 25 holdout subjects too.
INCLUDE_FINAL_25_IN_3C = False

print('3B:', SPLIT_FOLDER)
print('PCA:', PCA_OUTPUT_FOLDER)
print('Results:', RESULTS_FOLDER)


3B: /content/drive/MyDrive/EEG Shared/Processed_data/Kosachenko_All_Subjects/3B_Exact_40_25_Split
PCA: /content/drive/MyDrive/EEG Shared/Processed_data/Kosachenko_All_Subjects/3C_PCA_Outputs_Clean
Results: /content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA


## 3C — Fresh 3B → 24-row epochs → PCA → saved to Drive

This deliberately **does not reuse the old 3C caches**. It rereads the current repaired 3B files, verifies subject membership across train/validation/holdout, then saves one `pca_outputs.npz` per subject.


In [ ]:
pca_manifest = build_and_save_pca_from_exact3b(
    split_folder=SPLIT_FOLDER,
    pca_output_folder=PCA_OUTPUT_FOLDER,
    target_column=TARGET_COLUMN,
    epoch_rows=EPOCH_ROWS,
    pc_count=PC_COUNT,
    chunk_rows=CHUNK_ROWS,
    include_holdout_subjects=INCLUDE_FINAL_25_IN_3C,
)

display(pca_manifest.head())
print('Subjects saved:', len(pca_manifest))


Feature columns per row: 99
Flattened dimensions per epoch: 2376
Target: load_in_sequence

Reading DEVELOPMENT train...


KeyboardInterrupt: 

In [ ]:
holdout_pca_manifest = build_and_save_pca_from_exact3b(
    split_folder=SPLIT_FOLDER,
    pca_output_folder=PCA_OUTPUT_FOLDER,
    target_column=TARGET_COLUMN,
    epoch_rows=EPOCH_ROWS,
    pc_count=PC_COUNT,
    chunk_rows=50000,
    include_holdout_subjects=True,
)

Feature columns per row: 99
Flattened dimensions per epoch: 2376
Target: load_in_sequence

Reading DEVELOPMENT train...
  train.csv: chunk 5, usable epochs=9624
  train.csv: chunk 10, usable epochs=19240
  train.csv: chunk 15, usable epochs=28822
Reading DEVELOPMENT validation...
  validation.csv: chunk 5, usable epochs=9587
Reading DEVELOPMENT holdout...
  holdout.csv: chunk 5, usable epochs=9604

Development subjects verified: 40
development 1/40 sub-034 saved | train/val/hold=733/359/366 | PCs=20
development 2/40 sub-035 saved | train/val/hold=733/367/358 | PCs=20
development 3/40 sub-036 saved | train/val/hold=740/358/360 | PCs=20
development 4/40 sub-038 saved | train/val/hold=719/370/369 | PCs=20
development 5/40 sub-042 saved | train/val/hold=736/375/374 | PCs=20
development 6/40 sub-043 saved | train/val/hold=728/362/368 | PCs=20
development 7/40 sub-044 saved | train/val/hold=733/364/361 | PCs=20
development 8/40 sub-045 saved | train/val/hold=743/355/360 | PCs=20
development 

## 3D — KNN + Logistic + SVM using the SAVED PCA outputs

3D does not recompute PCA. Each subject gets its own three models. It saves MAE, Spearman, accuracy, and epoch-level target/predictions.


In [ ]:
dev_metrics, dev_predictions, dev_paths = run_models_from_saved_pca(
    pca_output_folder=PCA_OUTPUT_FOLDER,
    output_folder=RESULTS_FOLDER,
    subject_group='development',
)

display(dev_metrics.groupby(['model','split'])[['accuracy','mae','spearman']].mean().round(4))
display(dev_predictions.head(20))
print(dev_paths)


development: subject 1/40 sub-034 complete
development: subject 2/40 sub-035 complete
development: subject 3/40 sub-036 complete
development: subject 4/40 sub-038 complete
development: subject 5/40 sub-042 complete
development: subject 6/40 sub-043 complete
development: subject 7/40 sub-044 complete
development: subject 8/40 sub-045 complete
development: subject 9/40 sub-048 complete
development: subject 10/40 sub-051 complete
development: subject 11/40 sub-052 complete
development: subject 12/40 sub-053 complete
development: subject 13/40 sub-054 complete
development: subject 14/40 sub-055 complete
development: subject 15/40 sub-056 complete
development: subject 16/40 sub-058 complete
development: subject 17/40 sub-059 complete
development: subject 18/40 sub-062 complete
development: subject 19/40 sub-065 complete
development: subject 20/40 sub-067 complete
development: subject 21/40 sub-068 complete
development: subject 22/40 sub-069 complete
development: subject 23/40 sub-070 comple

accuracy     mae  spearman
model               split                                 
KNN                 holdout       0.1044  3.3148    0.0950
                    train         1.0000  0.0000    1.0000
                    validation    0.1014  3.3181    0.0928
Logistic Regression holdout       0.0876  4.1681    0.1203
                    train         0.1818  3.7986    0.2083
                    validation    0.0840  4.2075    0.1062
SVM                 holdout       0.1129  2.7486    0.2051
                    train         0.1788  2.4456    0.4521
                    validation    0.1083  2.7475    0.2102

model,subject_id,subject_group,split,epoch_uid,target,knn_prediction,logistic_prediction,svm_prediction
0,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_0,1.0,3.0,1.0,4.095531
1,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_10,2.0,3.0,7.0,4.420955
2,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_100,10.0,3.0,8.0,4.949632
3,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_101,11.0,1.0,4.0,4.770509
4,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1012,4.0,3.0,10.0,4.432783
5,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1022,1.0,3.0,3.0,3.568767
6,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1023,2.0,4.0,2.0,3.287990
7,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1024,3.0,5.0,11.0,6.145371
8,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1029,8.0,5.0,13.0,6.027581
9,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1030,9.0,8.0,8.0,7.664953


{'metrics': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/development_metrics.csv'), 'predictions_long': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/development_epoch_predictions_long.csv'), 'predictions_wide': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/development_epoch_predictions_wide.csv'), 'summary': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/development_summary.csv')}


## Final 25 holdout subjects — run only after hyperparameters are fixed

When you are ready, set `INCLUDE_FINAL_25_IN_3C = True`, rerun the 3C cell once so their PCA files are saved, then run the cell below.


In [ ]:
# FINAL 25 SUBJECTS

final_metrics, final_predictions, final_paths = run_models_from_saved_pca(
    pca_output_folder=PCA_OUTPUT_FOLDER,
    output_folder=RESULTS_FOLDER,
    subject_group='holdout_subjects',
)

display(
    final_metrics.groupby(['model','split'])[
        ['accuracy','mae','spearman']
    ].mean().round(4)
)

display(final_predictions.head(20))

print(final_paths)

all_six_path, all_six = combine_all_six_predictions(
    RESULTS_FOLDER
)

print('All six datasets saved to:', all_six_path)

display(all_six.head(20))

holdout_subjects: subject 1/25 sub-032 complete
holdout_subjects: subject 2/25 sub-033 complete
holdout_subjects: subject 3/25 sub-039 complete
holdout_subjects: subject 4/25 sub-040 complete
holdout_subjects: subject 5/25 sub-041 complete
holdout_subjects: subject 6/25 sub-046 complete
holdout_subjects: subject 7/25 sub-047 complete
holdout_subjects: subject 8/25 sub-049 complete
holdout_subjects: subject 9/25 sub-050 complete
holdout_subjects: subject 10/25 sub-057 complete
holdout_subjects: subject 11/25 sub-060 complete
holdout_subjects: subject 12/25 sub-061 complete
holdout_subjects: subject 13/25 sub-063 complete
holdout_subjects: subject 14/25 sub-064 complete
holdout_subjects: subject 15/25 sub-071 complete
holdout_subjects: subject 16/25 sub-072 complete
holdout_subjects: subject 17/25 sub-075 complete
holdout_subjects: subject 18/25 sub-078 complete
holdout_subjects: subject 19/25 sub-082 complete
holdout_subjects: subject 20/25 sub-084 complete
holdout_subjects: subject 21/

accuracy     mae  spearman
model               split                                 
KNN                 holdout       0.1068  3.3340    0.0830
                    train         1.0000  0.0000    1.0000
                    validation    0.1038  3.3475    0.0660
Logistic Regression holdout       0.0829  4.2094    0.1037
                    train         0.1798  3.8156    0.1911
                    validation    0.0828  4.2071    0.0956
SVM                 holdout       0.1085  2.7435    0.1992
                    train         0.1751  2.4653    0.4497
                    validation    0.1136  2.7529    0.1768

model,subject_id,subject_group,split,epoch_uid,target,knn_prediction,logistic_prediction,svm_prediction
0,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_0,1.0,9.0,9.0,5.208862
1,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1002,4.0,6.0,6.0,5.395505
2,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1005,7.0,5.0,13.0,5.503550
3,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_101,7.0,7.0,8.0,7.421311
4,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1011,13.0,5.0,8.0,8.101250
5,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1019,3.0,4.0,11.0,5.647083
6,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1020,4.0,1.0,2.0,4.421605
7,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1022,1.0,2.0,11.0,3.190446
8,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1023,2.0,1.0,1.0,2.220226
9,32,holdout_subjects,holdout,032__sub-032_task-memory_eeg__digit_epoch_1024,3.0,4.0,11.0,6.097570


{'metrics': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/holdout_subjects_metrics.csv'), 'predictions_long': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/holdout_subjects_epoch_predictions_long.csv'), 'predictions_wide': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/holdout_subjects_epoch_predictions_wide.csv'), 'summary': PosixPath('/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/holdout_subjects_summary.csv')}
All six datasets saved to: /content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/all_six_datasets_epoch_predictions.csv


,subject_id,subject_group,split,epoch_uid,target,knn_prediction,logistic_prediction,svm_prediction
0,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_0,1.0,3.0,1.0,4.095531
1,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_10,2.0,3.0,7.0,4.420955
2,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_100,10.0,3.0,8.0,4.949632
3,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_101,11.0,1.0,4.0,4.770509
4,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1012,4.0,3.0,10.0,4.432783
5,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1022,1.0,3.0,3.0,3.568767
6,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1023,2.0,4.0,2.0,3.287990
7,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1024,3.0,5.0,11.0,6.145371
8,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1029,8.0,5.0,13.0,6.027581
9,34,development,holdout,034__sub-034_task-memory_eeg__digit_epoch_1030,9.0,8.0,8.0,7.664953


In [ ]:
import pandas as pd

# ============================================================
# KNN — FOUR REQUIRED RESULT GROUPS
# ============================================================

all_metrics = pd.concat(
    [development_metrics, final_metrics],
    ignore_index=True
)

# KNN only
knn = all_metrics[
    all_metrics["model"] == "KNN"
].copy()

# Keep:
# development      -> train + validation
# holdout_subjects -> train + validation
knn = knn[
    (
        (knn["subject_group"] == "development")
        & (knn["split"].isin(["train", "validation"]))
    )
    |
    (
        (knn["subject_group"] == "holdout_subjects")
        & (knn["split"].isin(["train", "validation"]))
    )
].copy()


# ============================================================
# FRIENDLY NAMES
# ============================================================

def make_label(row):

    if (
        row["subject_group"] == "development"
        and row["split"] == "train"
    ):
        return "Training samples on training subjects"

    if (
        row["subject_group"] == "development"
        and row["split"] == "validation"
    ):
        return "Validation samples on training subjects"

    if (
        row["subject_group"] == "holdout_subjects"
        and row["split"] == "train"
    ):
        return "Training samples on holdout subjects"

    if (
        row["subject_group"] == "holdout_subjects"
        and row["split"] == "validation"
    ):
        return "Validation samples on holdout subjects"


knn["result_group"] = knn.apply(
    make_label,
    axis=1
)


# ============================================================
# SUMMARY
# ============================================================

order = [
    "Training samples on training subjects",
    "Validation samples on training subjects",
    "Training samples on holdout subjects",
    "Validation samples on holdout subjects",
]

summary = (
    knn
    .groupby("result_group")[
        ["accuracy", "mae", "spearman"]
    ]
    .mean()
    .reindex(order)
    .round(4)
)

print("=" * 80)
print("KNN — FOUR REQUIRED RESULTS")
print("=" * 80)

display(summary)


# ============================================================
# SUBJECT-BY-SUBJECT RESULTS
# ============================================================

subject_results = knn[
    [
        "subject_id",
        "result_group",
        "accuracy",
        "mae",
        "spearman",
    ]
].copy()

subject_results["result_group"] = pd.Categorical(
    subject_results["result_group"],
    categories=order,
    ordered=True
)

subject_results = subject_results.sort_values(
    ["result_group", "subject_id"]
)

print("\nSUBJECT-BY-SUBJECT KNN RESULTS")
display(subject_results)

KNN — FOUR REQUIRED RESULTS


,accuracy,mae,spearman
result_group,,,
Training samples on training subjects,1.0000,0.0000,1.0000
Validation samples on training subjects,0.1014,3.3181,0.0928
Training samples on holdout subjects,1.0000,0.0000,1.0000
Validation samples on holdout subjects,0.1038,3.3475,0.0660



SUBJECT-BY-SUBJECT KNN RESULTS


,subject_id,result_group,accuracy,mae,spearman
0,34,Training samples on training subjects,1.000000,0.000000,1.000000
9,35,Training samples on training subjects,1.000000,0.000000,1.000000
18,36,Training samples on training subjects,1.000000,0.000000,1.000000
27,38,Training samples on training subjects,1.000000,0.000000,1.000000
36,42,Training samples on training subjects,1.000000,0.000000,1.000000
...,...,...,...,...,...
541,86,Validation samples on holdout subjects,0.100543,3.230978,0.121421
550,89,Validation samples on holdout subjects,0.098361,3.311476,0.069666
559,92,Validation samples on holdout subjects,0.106101,3.419098,0.052626
568,94,Validation samples on holdout subjects,0.104615,3.110769,0.050839


In [ ]:
import pandas as pd

all_metrics = pd.concat(
    [dev_metrics, final_metrics],
    ignore_index=True
)
# SVM only
svm = all_metrics[
    all_metrics["model"] == "SVM"
].copy()

# Keep the four requested combinations
svm = svm[
    (
        (svm["subject_group"] == "development")
        & (svm["split"].isin(["train", "validation"]))
    )
    |
    (
        (svm["subject_group"] == "holdout_subjects")
        & (svm["split"].isin(["train", "validation"]))
    )
].copy()


def make_label(row):

    if (
        row["subject_group"] == "development"
        and row["split"] == "train"
    ):
        return "Training samples on training subjects"

    if (
        row["subject_group"] == "development"
        and row["split"] == "validation"
    ):
        return "Validation samples on training subjects"

    if (
        row["subject_group"] == "holdout_subjects"
        and row["split"] == "train"
    ):
        return "Training samples on holdout subjects"

    if (
        row["subject_group"] == "holdout_subjects"
        and row["split"] == "validation"
    ):
        return "Validation samples on holdout subjects"


svm["result_group"] = svm.apply(
    make_label,
    axis=1
)

order = [
    "Training samples on training subjects",
    "Validation samples on training subjects",
    "Training samples on holdout subjects",
    "Validation samples on holdout subjects",
]

# ============================================================
# SUMMARY
# ============================================================

summary = (
    svm
    .groupby("result_group")[
        ["accuracy", "mae", "spearman"]
    ]
    .mean()
    .reindex(order)
    .round(4)
)

print("=" * 80)
print("SVM — FOUR REQUIRED RESULTS")
print("=" * 80)

display(summary)


# ============================================================
# SUBJECT-BY-SUBJECT
# ============================================================

subject_results = svm[
    [
        "subject_id",
        "result_group",
        "accuracy",
        "mae",
        "spearman",
    ]
].copy()

subject_results["result_group"] = pd.Categorical(
    subject_results["result_group"],
    categories=order,
    ordered=True
)

subject_results = subject_results.sort_values(
    ["result_group", "subject_id"]
)

print("\nSUBJECT-BY-SUBJECT SVM RESULTS")
display(subject_results)

SVM — FOUR REQUIRED RESULTS


,accuracy,mae,spearman
result_group,,,
Training samples on training subjects,0.1788,2.4456,0.4521
Validation samples on training subjects,0.1083,2.7475,0.2102
Training samples on holdout subjects,0.1751,2.4653,0.4497
Validation samples on holdout subjects,0.1136,2.7529,0.1768



SUBJECT-BY-SUBJECT SVM RESULTS


,subject_id,result_group,accuracy,mae,spearman
6,34,Training samples on training subjects,0.197817,2.319301,0.504701
15,35,Training samples on training subjects,0.174625,2.325275,0.549067
24,36,Training samples on training subjects,0.175676,2.499698,0.438012
33,38,Training samples on training subjects,0.159944,2.544432,0.334128
42,42,Training samples on training subjects,0.141304,2.562253,0.357076
...,...,...,...,...,...
547,86,Validation samples on holdout subjects,0.089674,2.704958,0.246161
556,89,Validation samples on holdout subjects,0.139344,2.608256,0.267665
565,92,Validation samples on holdout subjects,0.071618,2.951894,0.097474
574,94,Validation samples on holdout subjects,0.104615,2.809646,0.101492


In [ ]:
from pathlib import Path
import pandas as pd

RESULTS_FOLDER = Path(
    "/content/drive/MyDrive/EEG Shared/Results/Luke/"
    "3D_Models_From_Clean_PCA"
)

# Load already-saved development results
development_metrics = pd.read_csv(
    RESULTS_FOLDER / "development_metrics.csv"
)

# Load final-25 results if they have already been run/saved
final_metrics_path = RESULTS_FOLDER / "holdout_subjects_metrics.csv"

if final_metrics_path.exists():
    final_metrics = pd.read_csv(final_metrics_path)
    print("Development + final metrics loaded.")
else:
    print("Development metrics loaded.")
    print("Final 25 metrics do not exist yet — run the final 25 models first.")

print("Development rows:", len(development_metrics))

if "final_metrics" in globals():
    print("Final rows:", len(final_metrics))

Development + final metrics loaded.
Development rows: 360
Final rows: 225


In [ ]:
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr


# ============================================================
# SVR + SAVED PCA TUNING
#
# ONLY DEVELOPMENT SUBJECTS ARE USED.
#
# Fit:        subject's TRAIN samples
# Select on:  subject's VALIDATION samples
#
# DO NOT use holdout subjects to choose hyperparameters.
# ============================================================


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PCA_OUTPUT_FOLDER = Path(
    "/content/drive/MyDrive/EEG Shared/Processed_data/"
    "Kosachenko_All_Subjects/3C_PCA_Outputs_Clean"
)

RESULTS_FOLDER = Path(
    "/content/drive/MyDrive/EEG Shared/Results/Luke/"
    "3D_Models_From_Clean_PCA"
)

RESULTS_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# HYPERPARAMETER SEARCH SPACE
# ------------------------------------------------------------

PC_COUNTS = [
    5,
    10,
    15,
    20,
]

C_VALUES = [
    0.1,
    1.0,
    10.0,
    100.0,
]

GAMMA_VALUES = [
    "scale",
    0.01,
    0.1,
]

EPSILON_VALUES = [
    0.05,
    0.1,
    0.5,
]


# ============================================================
# HELPERS
# ============================================================

def normalize_subject_id(value):

    s = str(value).strip()

    if s.endswith(".0"):
        s = s[:-2]

    if s.isdigit():
        return str(int(s))

    return s


def safe_spearman(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if len(y_true) < 2:
        return np.nan

    if np.all(y_true == y_true[0]):
        return np.nan

    if np.all(y_pred == y_pred[0]):
        return np.nan

    return float(
        spearmanr(
            y_true,
            y_pred
        ).statistic
    )


# ============================================================
# FIND ALL SAVED DEVELOPMENT SUBJECTS DIRECTLY FROM DRIVE
# ============================================================

development_folder = (
    PCA_OUTPUT_FOLDER /
    "development"
)

subject_files = sorted(
    development_folder.glob(
        "sub-*/pca_outputs.npz"
    ),
    key=lambda p: int(
        p.parent.name.replace(
            "sub-",
            ""
        )
    )
)


print(
    "Development PCA files found:",
    len(subject_files)
)


if len(subject_files) != 40:

    raise RuntimeError(
        f"Expected 40 development PCA files, "
        f"found {len(subject_files)}"
    )


subjects = []

for path in subject_files:

    subject_id = normalize_subject_id(
        path.parent.name.replace(
            "sub-",
            ""
        )
    )

    subjects.append(
        subject_id
    )


print(
    "Subjects:",
    [x.zfill(3) for x in subjects]
)


# ============================================================
# LOAD ALL DEVELOPMENT PCA DATA ONCE
#
# These files are small compared with the original CSVs,
# so this avoids repeatedly reading from Google Drive
# for every hyperparameter combination.
# ============================================================

subject_data = {}


for i, (
    subject,
    path
) in enumerate(
    zip(
        subjects,
        subject_files
    ),
    start=1
):

    data = np.load(
        path,
        allow_pickle=True
    )


    X_train = np.asarray(
        data["X_train_pca"],
        dtype=np.float32
    )

    y_train = np.asarray(
        data["y_train"],
        dtype=np.float32
    )


    X_validation = np.asarray(
        data["X_validation_pca"],
        dtype=np.float32
    )

    y_validation = np.asarray(
        data["y_validation"],
        dtype=np.float32
    )


    if len(X_train) == 0:
        raise RuntimeError(
            f"Subject {subject} "
            f"has empty training data."
        )


    if len(X_validation) == 0:
        raise RuntimeError(
            f"Subject {subject} "
            f"has empty validation data."
        )


    subject_data[
        subject
    ] = {
        "X_train":
            X_train,

        "y_train":
            y_train,

        "X_validation":
            X_validation,

        "y_validation":
            y_validation,
    }


    print(
        f"Loaded {i}/40 "
        f"sub-{subject.zfill(3)} | "
        f"train={len(X_train)} | "
        f"validation={len(X_validation)} | "
        f"saved PCs={X_train.shape[1]}"
    )


# ============================================================
# VERIFY MAXIMUM AVAILABLE PCA COMPONENTS
# ============================================================

min_saved_pc_count = min(
    d["X_train"].shape[1]
    for d in subject_data.values()
)


print(
    "\nMinimum saved PCs across subjects:",
    min_saved_pc_count
)


invalid_pc_counts = [
    n
    for n in PC_COUNTS
    if n > min_saved_pc_count
]


if invalid_pc_counts:

    raise RuntimeError(
        f"Requested PC counts "
        f"{invalid_pc_counts}, "
        f"but only "
        f"{min_saved_pc_count} "
        f"components are available "
        f"for every subject."
    )


# ============================================================
# CREATE ALL COMBINATIONS
# ============================================================

combinations = list(
    product(
        PC_COUNTS,
        C_VALUES,
        GAMMA_VALUES,
        EPSILON_VALUES,
    )
)


print(
    "\nCombinations:",
    len(combinations)
)

print(
    "Subjects per combination:",
    len(subjects)
)

print(
    "Total SVR fits:",
    len(combinations)
    *
    len(subjects)
)

print(
    "\nStarting tuning...\n"
)


# ============================================================
# TUNING LOOP
# ============================================================

result_rows = []

subject_result_rows = []


for combo_number, (
    pc_count,
    C,
    gamma,
    epsilon,
) in enumerate(
    combinations,
    start=1
):


    combo_spearman = []
    combo_mae = []
    combo_accuracy = []


    for subject in subjects:

        d = subject_data[
            subject
        ]


        # ----------------------------------------------------
        # KEEP ONLY FIRST N PCA COMPONENTS
        # ----------------------------------------------------

        X_train = (
            d["X_train"][
                :,
                :pc_count
            ]
        )

        X_validation = (
            d["X_validation"][
                :,
                :pc_count
            ]
        )

        y_train = d[
            "y_train"
        ]

        y_validation = d[
            "y_validation"
        ]


        # ----------------------------------------------------
        # FIT SVR
        # ----------------------------------------------------

        model = SVR(
            kernel="rbf",
            C=C,
            gamma=gamma,
            epsilon=epsilon,
        )


        model.fit(
            X_train,
            y_train
        )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        prediction = (
            model.predict(
                X_validation
            )
        )


        rho = safe_spearman(
            y_validation,
            prediction
        )


        mae = float(
            mean_absolute_error(
                y_validation,
                prediction
            )
        )


        # Rounded exact-match accuracy
        rounded_prediction = np.clip(
            np.rint(
                prediction
            ),
            1,
            13
        ).astype(int)


        accuracy = float(
            np.mean(
                rounded_prediction
                ==
                y_validation.astype(int)
            )
        )


        if not np.isnan(rho):

            combo_spearman.append(
                rho
            )


        combo_mae.append(
            mae
        )

        combo_accuracy.append(
            accuracy
        )


        # Save subject-level tuning result
        subject_result_rows.append(
            {
                "pc_count":
                    pc_count,

                "C":
                    C,

                "gamma":
                    gamma,

                "epsilon":
                    epsilon,

                "subject_id":
                    subject,

                "validation_spearman":
                    rho,

                "validation_mae":
                    mae,

                "validation_accuracy":
                    accuracy,
            }
        )


    # ========================================================
    # AGGREGATE ACROSS 40 SUBJECTS
    # ========================================================

    mean_spearman = float(
        np.nanmean(
            combo_spearman
        )
    )

    median_spearman = float(
        np.nanmedian(
            combo_spearman
        )
    )

    mean_mae = float(
        np.mean(
            combo_mae
        )
    )

    mean_accuracy = float(
        np.mean(
            combo_accuracy
        )
    )


    result_rows.append(
        {
            "pc_count":
                pc_count,

            "C":
                C,

            "gamma":
                gamma,

            "epsilon":
                epsilon,

            "mean_validation_spearman":
                mean_spearman,

            "median_validation_spearman":
                median_spearman,

            "mean_validation_mae":
                mean_mae,

            "mean_validation_accuracy":
                mean_accuracy,

            "subjects":
                len(subjects),
        }
    )


    print(
        f"{combo_number:3d}/"
        f"{len(combinations)} | "
        f"PC={pc_count:2d} | "
        f"C={C:<6} | "
        f"gamma={str(gamma):<6} | "
        f"eps={epsilon:<4} | "
        f"rho={mean_spearman:.4f} | "
        f"MAE={mean_mae:.4f} | "
        f"acc={mean_accuracy:.4f}"
    )


# ============================================================
# SUMMARY TABLE
# ============================================================

tuning_results = pd.DataFrame(
    result_rows
)


# PRIMARY:
# highest validation Spearman
#
# SECONDARY:
# lowest validation MAE

tuning_results = (
    tuning_results
    .sort_values(
        [
            "mean_validation_spearman",
            "mean_validation_mae",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "TOP 20 SVR + PCA SETTINGS"
)

print(
    "=" * 100
)


display(
    tuning_results.head(20)
)


# ============================================================
# BEST RESULT
# ============================================================

best = tuning_results.iloc[0]


print(
    "\n"
    + "=" * 100
)

print(
    "BEST SETTINGS"
)

print(
    "=" * 100
)


print(
    "PC count:",
    int(
        best[
            "pc_count"
        ]
    )
)

print(
    "C:",
    best["C"]
)

print(
    "gamma:",
    best["gamma"]
)

print(
    "epsilon:",
    best["epsilon"]
)


print(
    "\nMean validation Spearman:",
    round(
        best[
            "mean_validation_spearman"
        ],
        4
    )
)

print(
    "Median validation Spearman:",
    round(
        best[
            "median_validation_spearman"
        ],
        4
    )
)

print(
    "Mean validation MAE:",
    round(
        best[
            "mean_validation_mae"
        ],
        4
    )
)

print(
    "Rounded validation accuracy:",
    round(
        best[
            "mean_validation_accuracy"
        ],
        4
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

summary_path = (
    RESULTS_FOLDER /
    "svr_pca_tuning_summary.csv"
)

subject_path = (
    RESULTS_FOLDER /
    "svr_pca_tuning_subject_results.csv"
)


tuning_results.to_csv(
    summary_path,
    index=False
)


pd.DataFrame(
    subject_result_rows
).to_csv(
    subject_path,
    index=False
)


print(
    "\nSaved tuning summary:"
)

print(
    summary_path
)


print(
    "\nSaved subject-level tuning results:"
)

print(
    subject_path
)

Development PCA files found: 40
Subjects: ['034', '035', '036', '038', '042', '043', '044', '045', '048', '051', '052', '053', '054', '055', '056', '058', '059', '062', '065', '067', '068', '069', '070', '073', '074', '076', '077', '079', '080', '081', '083', '085', '087', '088', '090', '091', '093', '095', '096', '097']
Loaded 1/40 sub-034 | train=733 | validation=359 | saved PCs=20
Loaded 2/40 sub-035 | train=733 | validation=367 | saved PCs=20
Loaded 3/40 sub-036 | train=740 | validation=358 | saved PCs=20
Loaded 4/40 sub-038 | train=719 | validation=370 | saved PCs=20
Loaded 5/40 sub-042 | train=736 | validation=375 | saved PCs=20
Loaded 6/40 sub-043 | train=728 | validation=362 | saved PCs=20
Loaded 7/40 sub-044 | train=733 | validation=364 | saved PCs=20
Loaded 8/40 sub-045 | train=743 | validation=355 | saved PCs=20
Loaded 9/40 sub-048 | train=731 | validation=367 | saved PCs=20
Loaded 10/40 sub-051 | train=721 | validation=369 | saved PCs=20
Loaded 11/40 sub-052 | train=731 | v

,pc_count,C,gamma,epsilon,mean_validation_spearman,median_validation_spearman,mean_validation_mae,mean_validation_accuracy,subjects
0,20,1.0,scale,0.10,0.210212,0.205948,2.747458,0.108308,40
1,20,1.0,scale,0.05,0.209817,0.206450,2.747788,0.107950,40
2,20,1.0,scale,0.50,0.209124,0.216771,2.746677,0.108830,40
3,15,1.0,scale,0.50,0.199955,0.214312,2.754885,0.107249,40
4,15,1.0,scale,0.10,0.197914,0.211205,2.756898,0.107310,40
5,15,1.0,scale,0.05,0.197229,0.206639,2.757238,0.106983,40
6,20,10.0,scale,0.50,0.193798,0.205604,2.830114,0.104995,40
7,20,10.0,scale,0.10,0.192012,0.199460,2.841720,0.105880,40
8,20,10.0,scale,0.05,0.191973,0.198859,2.842826,0.105801,40
9,20,0.1,scale,0.05,0.187815,0.188343,2.785527,0.113726,40



BEST SETTINGS
PC count: 20
C: 1.0
gamma: scale
epsilon: 0.1

Mean validation Spearman: 0.2102
Median validation Spearman: 0.2059
Mean validation MAE: 2.7475
Rounded validation accuracy: 0.1083

Saved tuning summary:
/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/svr_pca_tuning_summary.csv

Saved subject-level tuning results:
/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/svr_pca_tuning_subject_results.csv


In [ ]:
# ============================================================
# EXPERIMENT 2 — SVR POLYNOMIAL KERNEL, DEGREE 3
# Keep everything else the same as Entry 1
# ============================================================

from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr
import numpy as np
import pandas as pd


results = []


for subject in subjects:

    d = subject_data[subject]

    # Same 20 PCs as Entry 1
    X_train = d["X_train"][:, :20]
    y_train = d["y_train"]

    X_validation = d["X_validation"][:, :20]
    y_validation = d["y_validation"]


    # --------------------------------------------------------
    # POLYNOMIAL SVR
    # --------------------------------------------------------

    model = SVR(
        kernel="poly",
        degree=3,
        C=1.0,
        gamma="scale",
        epsilon=0.1,
    )

    model.fit(
        X_train,
        y_train
    )


    # ========================================================
    # TRAINING RESULTS
    # ========================================================

    train_pred = model.predict(
        X_train
    )

    train_rounded = np.clip(
        np.rint(train_pred),
        1,
        13
    ).astype(int)

    train_accuracy = np.mean(
        train_rounded == y_train.astype(int)
    )

    train_mae = mean_absolute_error(
        y_train,
        train_pred
    )

    train_spearman = spearmanr(
        y_train,
        train_pred
    ).statistic


    # ========================================================
    # VALIDATION RESULTS
    # ========================================================

    val_pred = model.predict(
        X_validation
    )

    val_rounded = np.clip(
        np.rint(val_pred),
        1,
        13
    ).astype(int)

    val_accuracy = np.mean(
        val_rounded == y_validation.astype(int)
    )

    val_mae = mean_absolute_error(
        y_validation,
        val_pred
    )

    val_spearman = spearmanr(
        y_validation,
        val_pred
    ).statistic


    results.append({
        "subject_id": subject,

        "train_accuracy": train_accuracy,
        "train_mae": train_mae,
        "train_spearman": train_spearman,

        "validation_accuracy": val_accuracy,
        "validation_mae": val_mae,
        "validation_spearman": val_spearman,
    })


# ============================================================
# RESULTS
# ============================================================

poly_results = pd.DataFrame(results)


summary = pd.DataFrame({
    "accuracy": [
        poly_results["train_accuracy"].mean(),
        poly_results["validation_accuracy"].mean(),
    ],

    "mae": [
        poly_results["train_mae"].mean(),
        poly_results["validation_mae"].mean(),
    ],

    "spearman": [
        poly_results["train_spearman"].mean(),
        poly_results["validation_spearman"].mean(),
    ],
}, index=[
    "Training samples on training subjects",
    "Validation samples on training subjects",
])


print("=" * 80)
print("SVR — POLYNOMIAL KERNEL DEGREE 3")
print("=" * 80)

display(
    summary.round(4)
)


print("\nSUBJECT-BY-SUBJECT RESULTS")

display(
    poly_results.round(4)
)


# ============================================================
# SAVE
# ============================================================

save_path = (
    RESULTS_FOLDER /
    "svr_poly_degree3_results.csv"
)

poly_results.to_csv(
    save_path,
    index=False
)

print("\nSaved to:")
print(save_path)

SVR — POLYNOMIAL KERNEL DEGREE 3


,accuracy,mae,spearman
Training samples on training subjects,0.2249,2.3751,0.4845
Validation samples on training subjects,0.1094,31.2545,0.1428



SUBJECT-BY-SUBJECT RESULTS


,subject_id,train_accuracy,train_mae,train_spearman,validation_accuracy,validation_mae,validation_spearman
0,34,0.2824,2.1128,0.5981,0.1393,2.7467,0.2201
1,35,0.2251,2.3505,0.5917,0.1144,2.7779,0.2261
2,36,0.2405,2.3026,0.4455,0.0894,3.2106,0.0747
3,38,0.1766,2.5166,0.4151,0.0919,8.6082,0.1113
4,42,0.2065,2.4863,0.3194,0.1067,3.1178,0.0521
5,43,0.1868,2.4835,0.4830,0.1188,3.4643,0.1385
6,44,0.2156,2.3648,0.5351,0.0879,2.9200,0.1908
7,45,0.2988,2.1938,0.5913,0.0873,2.8381,0.1292
8,48,0.2599,2.1990,0.6100,0.1117,966.5524,0.1146
9,51,0.2552,2.2129,0.6540,0.0921,3.1358,0.2536



Saved to:
/content/drive/MyDrive/EEG Shared/Results/Luke/3D_Models_From_Clean_PCA/svr_poly_degree3_results.csv
